# Using the data for network building

In [1]:
import pandas as pd
from discovery_utils.utils.llm import batch_check

import pandas as pd
import os

# Alternative: s3://discovery-iss/data/afs_scanning/afs_open_alex_scan_parenting_interventions.csv
INPUT_DATA = 'afs_open_alex_scan_parenting_interventions.csv'

data_df = (
    pd.read_csv(INPUT_DATA)
    .query("publication_year >= 2000")
    .assign(text = lambda df: df['title'] + ' ' + df['abstract'])
)
len(data_df)

# testing on a smaller sample for now
# data_df = data_df.sample(100, random_state=42)

34145

In [2]:
relevant_df = (
    pd.read_json("afs_open_alex_scan_2.jsonl", lines=True)
    .query("is_relevant == 'yes'")
)

In [3]:
from ast import literal_eval

def extract_author_countries(authorships):
    try:
        # Convert the string representation of the list to an actual list
        authors = literal_eval(authorships)
        countries = []
        for author in authors:
            for institution in author.get('institutions', []):
                if 'country_code' in institution:
                    countries.append(institution['country_code'])
        # remove null values
        countries = [country for country in countries if country is not None]
        return sorted(list(set(countries)))
    except (ValueError, SyntaxError):
        # Handle the case where the string cannot be evaluated
        return []

## Run the LLM pipeline

In [4]:
_data_df = data_df.query("id in @relevant_df.id.tolist()")
ids = _data_df.id.tolist()
text = data_df.text.tolist()
test_data = dict(zip(ids, text))

In [5]:
len(test_data)

3839

In [6]:
system_message_pass1 = """
    Extract structured information about the main results of the intervention from the provided text.

    The intervention could be relevant to developmental, behavioural, or well-being outcomes
    such as children language development, cognitive development, social-emotional skills, physical and mental health, or
    parental outcomes such as confidence, skills, behaviours, knowledge, self-efficacy, and well-being.
    
    Unless requested otherwise, adhere as precisely as possible to the language and text that is used in the provided text document.
    
    If the requested information is not described, return N/A. DO NOT make up any false information or false inferences.
"""

fields_pass1 = [
    # Relevance
    # {"name": "main_results", "type": "list[str]", "description": "List of each of the main results of the study, for example: ['parenting education decreased child mental health problems',...]"},
    {"name": "main_results", "type": "list[str]", "description": "Succinctly extract the main intervention and its result of the study; an intervention means that there was an intervention or predictor variable that was manipulated, and an outcome variable - succinctly describe those in one short sentence. If there are more than one main intervention, extract more - but extract only those that are the focus of the paper."},
    {"name": "sample_size", "type": "list[int]", "description": "Number of participants in the study. If there are multiple samples, provide the sample size for each."},
]

In [7]:
processor = batch_check.LLMProcessor(
    model_name="gpt-4.1-mini",
    # model_name="gpt-4o-mini",
    temperature=0,
    output_path="pet_scan_3.jsonl",
    system_message=system_message_pass1,
    session_name="pet_scan",
    output_fields=fields_pass1,
)

processor.run(test_data, batch_size=30, sleep_time=0.5)

2025-05-20 00:33:35,219 - root - INFO - Using OpenAI


<Task pending name='Task-5' coro=<LLMProcessor.process_text_data() running at /Users/karlis.kanders/Code/discovery_utils/discovery_utils/utils/llm/batch_check.py:120>>

2025-05-20 00:33:35,297 - root - INFO - Processing batch 1/123
2025-05-20 00:33:39,713 - root - INFO - Processing batch 2/123
2025-05-20 00:33:46,472 - root - INFO - Processing batch 3/123
2025-05-20 00:33:50,400 - root - INFO - Processing batch 4/123
2025-05-20 00:33:55,705 - root - INFO - Processing batch 5/123
2025-05-20 00:33:59,456 - root - INFO - Processing batch 6/123
2025-05-20 00:34:06,367 - root - INFO - Processing batch 7/123
2025-05-20 00:34:10,243 - root - INFO - Processing batch 8/123
2025-05-20 00:34:14,951 - root - INFO - Processing batch 9/123
2025-05-20 00:34:18,754 - root - INFO - Processing batch 10/123
2025-05-20 00:34:22,715 - root - INFO - Processing batch 11/123
2025-05-20 00:34:27,309 - root - INFO - Processing batch 12/123
2025-05-20 00:34:32,417 - root - INFO - Processing batch 13/123
2025-05-20 00:34:36,003 - root - INFO - Processing batch 14/123
2025-05-20 00:34:39,050 - root - INFO - Processing batch 15/123
2025-05-20 00:34:43,409 - root - INFO - Processin

In [8]:
output_df = pd.read_json("pet_scan_3.jsonl", lines=True)
output_df

,main_results,sample_size,id,timestamp,model,temperature
0,[Implementation of nationwide school closures ...,[220000000],https://openalex.org/W2160124959,2025-05-19 23:00:33.588303+00:00,gpt-4.1-mini,0
1,[Implementation of a coproduced family centere...,"[3106, 2148, 435, 203, 586]",https://openalex.org/W1982264918,2025-05-19 23:00:33.590514+00:00,gpt-4.1-mini,0
2,[Early Head Start improved 3-year-old children...,[3001],https://openalex.org/W2066202980,2025-05-19 23:00:33.590615+00:00,gpt-4.1-mini,0
3,[Home- and center-based programs with a parent...,[0],https://openalex.org/W2116341692,2025-05-19 23:00:33.590684+00:00,gpt-4.1-mini,0
4,[Parent-implemented language interventions sig...,[18],https://openalex.org/W2058887279,2025-05-19 23:00:33.590746+00:00,gpt-4.1-mini,0
...,...,...,...,...,...,...
3834,"[Infants engage in rhythmic, musical-like dial...",[],https://openalex.org/W2030543951,2025-05-19 23:42:26.497308+00:00,gpt-4.1-mini,0
3835,[Assertive community treatment (ACT) tailored ...,[35],https://openalex.org/W2149642767,2025-05-19 23:42:26.497440+00:00,gpt-4.1-mini,0
3836,[An individualized DIR/FloortimeTM-based inter...,[25],https://openalex.org/W1973779798,2025-05-19 23:42:26.497577+00:00,gpt-4.1-mini,0
3837,[The study identified substantial and heteroge...,[44],https://openalex.org/W2145116009,2025-05-19 23:42:26.497716+00:00,gpt-4.1-mini,0


In [9]:
from pydantic import BaseModel, Field
from discovery_utils.utils.llm import llm_utils
import importlib
import tiktoken
importlib.reload(tiktoken)
importlib.reload(llm_utils)

prompt = {
    "system_message": """
        You are an expert researcher.
        Extract structured information about the intervention result from the provided text.
        Unless requested otherwise, adhere as precisely as possible to the language and text that is used in the provided text document.
        Keep your extractions as short as possible, but include all relevant information.
        If the requested information is not described, return N/A. DO NOT make up any false information or false inferences.
    """,
    "user_message": """
        The specific intervention result we are interested in is the following: {result}
        \n\n
        The text is the following: {input}
    """,
}

class ResultModel(BaseModel):
    intervention_variable: str = Field(description="The intervention or predictor variable (i.e., what was manipulated or used as a predictor)")
    outcome_variable: str = Field(description="The outcome variable (i.e., what was measured or predicted)")
    effect_size: str = Field(description="The effect size of the intervention (if available, otherwise N/A)")
    effect_size_number: float = Field(description="The effect size of the intervention (if available, otherwise N/A)")
    effect_size_type: str = Field(description="The effect size type for this main result (e.g. odds ratio, difference of means)")
    uncertainty: str = Field(description="The estimate of uncertainty in the effect size (e.g. s.e., 95% CI)")
    p_value: str = Field(description="The p-value of the effect size (if available, otherwise N/A)")
    effect_sign: str = Field(description="The direction of the effect, choose ONE of the following: intervention increases the outcome=POSITIVE, intervention decreases the outcome=NEGATIVE, no effect=NONE, unknown=N/A)")

Generator = llm_utils.StructuredOutputGenerator(
    model_dict={"model_name": "gpt-4.1-mini", "temperature": 0, "max_tokens": 120000},
    output_class=ResultModel,
    prompts=prompt,
    check_token_length=False,
)


def extract_result_data(result: str, text: str):
    """Classify the relevance of a text to a given mission.

    Uses mission scope definitions in the configuration file
    Args:
        input: The input text to classify.
        mission: The mission to classify the text against.

    Returns:
        RelevanceClassifier: The classification result.
    """
    return Generator.generate({"result": result, "input": text})


2025-05-20 00:45:36,244 - root - INFO - Using OpenAI


In [14]:
result_datas = []
for i, row in output_df.iterrows():
    if i <= 1714:
        continue
    print(f"{i}/{len(output_df)}")
    main_results = row['main_results']
    for result in main_results:
        result_data = extract_result_data(result, test_data[row['id']])
        result_datas.append({"id": row['id'], "result": result, **result_data.model_dump()})
result_df = pd.DataFrame(result_datas)


1715/3839
1716/3839
1717/3839
1718/3839
1719/3839
1720/3839
1721/3839
1722/3839
1723/3839
1724/3839
1725/3839
1726/3839
1727/3839
1728/3839
1729/3839
1730/3839
1731/3839
1732/3839
1733/3839
1734/3839
1735/3839
1736/3839
1737/3839
1738/3839
1739/3839
1740/3839
1741/3839
1742/3839
1743/3839
1744/3839
1745/3839
1746/3839
1747/3839
1748/3839
1749/3839
1750/3839
1751/3839
1752/3839
1753/3839
1754/3839
1755/3839
1756/3839
1757/3839
1758/3839
1759/3839
1760/3839
1761/3839
1762/3839
1763/3839
1764/3839
1765/3839
1766/3839
1767/3839
1768/3839
1769/3839
1770/3839
1771/3839
1772/3839
1773/3839
1774/3839
1775/3839
1776/3839
1777/3839
1778/3839
1779/3839
1780/3839
1781/3839
1782/3839
1783/3839
1784/3839
1785/3839
1786/3839
1787/3839
1788/3839
1789/3839
1790/3839
1791/3839
1792/3839
1793/3839
1794/3839
1795/3839
1796/3839
1797/3839
1798/3839
1799/3839
1800/3839
1801/3839
1802/3839
1803/3839
1804/3839
1805/3839
1806/3839
1807/3839
1808/3839
1809/3839
1810/3839
1811/3839
1812/3839
1813/3839
1814/3839


2025-05-20 07:58:54,997 - openai._base_client - INFO - Retrying request to /chat/completions in 0.414620 seconds


1969/3839
1970/3839
1971/3839
1972/3839
1973/3839
1974/3839
1975/3839
1976/3839
1977/3839
1978/3839
1979/3839
1980/3839
1981/3839
1982/3839
1983/3839
1984/3839
1985/3839
1986/3839
1987/3839
1988/3839
1989/3839
1990/3839
1991/3839
1992/3839
1993/3839
1994/3839
1995/3839
1996/3839
1997/3839
1998/3839
1999/3839
2000/3839
2001/3839
2002/3839
2003/3839
2004/3839
2005/3839
2006/3839
2007/3839
2008/3839
2009/3839
2010/3839
2011/3839
2012/3839
2013/3839
2014/3839
2015/3839
2016/3839
2017/3839
2018/3839
2019/3839
2020/3839
2021/3839
2022/3839
2023/3839
2024/3839
2025/3839
2026/3839
2027/3839
2028/3839
2029/3839
2030/3839
2031/3839
2032/3839
2033/3839
2034/3839
2035/3839
2036/3839
2037/3839
2038/3839
2039/3839
2040/3839
2041/3839
2042/3839
2043/3839
2044/3839
2045/3839
2046/3839
2047/3839
2048/3839
2049/3839
2050/3839
2051/3839
2052/3839
2053/3839
2054/3839
2055/3839
2056/3839
2057/3839
2058/3839
2059/3839
2060/3839
2061/3839
2062/3839
2063/3839
2064/3839
2065/3839
2066/3839
2067/3839
2068/3839


LengthFinishReasonError: Could not parse response content as the length limit was reached - CompletionUsage(completion_tokens=59, prompt_tokens=742, total_tokens=801, completion_tokens_details=CompletionTokensDetails(accepted_prediction_tokens=0, audio_tokens=0, reasoning_tokens=0, rejected_prediction_tokens=0), prompt_tokens_details=PromptTokensDetails(audio_tokens=0, cached_tokens=0))

In [15]:
result_df = pd.DataFrame(result_datas)

In [19]:
result_df[result_df.effect_size_type.str.contains("effect size")]

,id,result,intervention_variable,outcome_variable,effect_size,effect_size_number,effect_size_type,uncertainty,p_value,effect_sign
8,https://openalex.org/W1951736428,Linguistic comprehension instruction showed sm...,Linguistic comprehension instruction,generalized linguistic comprehension outcomes ...,small positive immediate effects on generalize...,0.200,standardized effect size (small to moderate),N/A,N/A,POSITIVE
9,https://openalex.org/W1951736428,Linguistic comprehension instruction had negli...,Linguistic comprehension instruction,generalized reading comprehension outcomes,negligible immediate effects,0.000,effect size (not numerically specified),few studies reporting follow-up assessments wi...,N/A,NONE
129,https://openalex.org/W2050802668,A 24-session caregiver-mediated joint attentio...,24-session caregiver-mediated joint attention ...,"joint engagement, responsiveness to joint atte...",medium to large effect sizes,0.000,effect size,N/A,significant improvements,POSITIVE
207,https://openalex.org/W2277832614,Level 4 SSTP was strongly supported as an effe...,Level 4 Stepping Stones Triple P (SSTP),child and parent outcomes in families of child...,Strong support as an effective intervention,0.700,effect size (d) for parenting style as an exam...,N/A,N/A,POSITIVE
233,https://openalex.org/W1578640603,Group social skills interventions (GSSIs) for ...,Group social skills interventions (GSSIs),social responsiveness as measured by the paren...,Large positive effect sizes,0.000,effect size,N/A,N/A,POSITIVE
234,https://openalex.org/W1578640603,GSSIs that included parent-groups and had grea...,GSSIs that include parent-groups and have grea...,social responsiveness (measured by SRS),larger effect sizes,0.000,effect size,N/A,N/A,POSITIVE
463,https://openalex.org/W1894556437,A comprehensive psychosocial intervention targ...,comprehensive psychosocial intervention target...,five of seven primary outcome measures in chil...,Standardized effect size estimates were predom...,0.000,standardized effect size,N/A,significant treatment effects,POSITIVE
464,https://openalex.org/W1894556437,The intervention's response-cost program effec...,response-cost program,problem behaviors and skills acquisition,Standardized effect size estimates were predom...,0.000,standardized effect size,N/A,N/A,POSITIVE
492,https://openalex.org/W2969935787,A 6-session family-based HIV prevention interv...,6-session family-based HIV prevention interven...,parents' reports of sex communication outcomes...,medium to large effects at postintervention an...,0.500,effect size (Cohen's d or similar),N/A,N/A,POSITIVE
493,https://openalex.org/W2969935787,Youth reports showed small to medium effects f...,family-based HIV prevention intervention targe...,communication variables (content and quality),small to medium effects,0.300,effect size (Cohen's d or similar),lasting through the 6-month follow-up period,N/A,POSITIVE


In [17]:
result_df.to_csv("pet_scan_3_results_2.csv", index=False)